## Project Requirements

The Income dataset contains a number of variables from the US census. The data file provided has a header describing the fields in the file, and approximately 32000 rows of data. 

Create a classifier using Spark + MLLib that can determine whether a person earns above $50k given the other variables in the data set. 

Note that you will need to convert any categorical fields into indexes in a similar way to what was done for the species variable in the Iris dataset.
 
Use the Random Forest and Decision Tree classifiers, and should be aiming to get a fairly high accuracy (>75%)


## Imports

In [86]:
from pyspark.sql import SparkSession 
import pyspark.sql.functions as F  
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

## Creating a  spark session

In [87]:
spark  = SparkSession.builder.appName("Income").getOrCreate()
# Check Spark Session Information
spark

#spark.stop() 

## Data Loading

In [88]:
#load the data
df = spark.read.csv("income.csv", header=True, inferSchema=True)
df.show(5)

+---+-----------------+--------+----------+---------------+-------------------+------------------+--------------+------+-------+------------+------------+--------------+--------------+------------+
|age|        workclass|  weight| education|education_years|     marital_status|        occupation|  relationship|  race|    sex|capital_gain|capital_loss|hours_per_week|   citizenship|income_class|
+---+-----------------+--------+----------+---------------+-------------------+------------------+--------------+------+-------+------------+------------+--------------+--------------+------------+
| 39|        State-gov| 77516.0| Bachelors|           13.0|      Never-married|      Adm-clerical| Not-in-family| White|   Male|      2174.0|         0.0|          40.0| United-States|       <=50K|
| 50| Self-emp-not-inc| 83311.0| Bachelors|           13.0| Married-civ-spouse|   Exec-managerial|       Husband| White|   Male|         0.0|         0.0|          13.0| United-States|       <=50K|
| 38|     

## Data Cleaning

In [89]:
#print the schema of the dataframe
df.printSchema()

root
 |-- age: integer (nullable = true)
 |-- workclass: string (nullable = true)
 |-- weight: double (nullable = true)
 |-- education: string (nullable = true)
 |-- education_years: double (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- relationship: string (nullable = true)
 |-- race: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- capital_gain: double (nullable = true)
 |-- capital_loss: double (nullable = true)
 |-- hours_per_week: double (nullable = true)
 |-- citizenship: string (nullable = true)
 |-- income_class: string (nullable = true)



### Check for missing values/nulls

In [90]:
# Count of nulls per column
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+---+---------+------+---------+---------------+--------------+----------+------------+----+---+------------+------------+--------------+-----------+------------+
|age|workclass|weight|education|education_years|marital_status|occupation|relationship|race|sex|capital_gain|capital_loss|hours_per_week|citizenship|income_class|
+---+---------+------+---------+---------------+--------------+----------+------------+----+---+------------+------------+--------------+-----------+------------+
|  0|        0|     0|        0|              0|             0|         0|           0|   0|  0|           0|           0|             0|          0|           0|
+---+---------+------+---------+---------------+--------------+----------+------------+----+---+------------+------------+--------------+-----------+------------+



### Check for duplicates

In [91]:
#check for duplicates
df.groupBy(df.columns).count().filter(F.col("count") > 1).show()


+---+-----------------+--------+-------------+---------------+-------------------+------------------+--------------+-------------------+-------+------------+------------+--------------+--------------+------------+-----+
|age|        workclass|  weight|    education|education_years|     marital_status|        occupation|  relationship|               race|    sex|capital_gain|capital_loss|hours_per_week|   citizenship|income_class|count|
+---+-----------------+--------+-------------+---------------+-------------------+------------------+--------------+-------------------+-------+------------+------------+--------------+--------------+------------+-----+
| 90|          Private| 52386.0| Some-college|           10.0|      Never-married|     Other-service| Not-in-family| Asian-Pac-Islander|   Male|         0.0|         0.0|          35.0| United-States|       <=50K|    2|
| 19|          Private|146679.0| Some-college|           10.0|      Never-married|   Exec-managerial|     Own-child|    

### Drop the duplicates

In [92]:
#remove duplicates
df_clean = df.distinct() 

#check count of rows before and after cleaning
print("Count of rows before cleaning: ", df.count())
print("Count of rows after cleaning: ", df_clean.count())

Count of rows before cleaning:  32561
Count of rows after cleaning:  32537


### Check for Class Distribution

In [93]:
#select string columns only
categorical_cols = [c for c, t in df_clean.dtypes if t == 'string']
categorical_cols

['workclass',
 'education',
 'marital_status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'citizenship',
 'income_class']

In [94]:
#check count of each distinct category ind the unindexed columns
for col in categorical_cols:
    df_clean.groupBy(col).count().show()

+-----------------+-----+
|        workclass|count|
+-----------------+-----+
|        State-gov| 1298|
|      Federal-gov|  960|
| Self-emp-not-inc| 2540|
|        Local-gov| 2093|
|          Private|22673|
|                ?| 1836|
|     Self-emp-inc| 1116|
|      Without-pay|   14|
|     Never-worked|    7|
+-----------------+-----+

+-------------+-----+
|    education|count|
+-------------+-----+
|  Prof-school|  576|
|         10th|  933|
|      7th-8th|  645|
|      5th-6th|  332|
|   Assoc-acdm| 1067|
|    Assoc-voc| 1382|
|      Masters| 1722|
|         12th|  433|
|    Preschool|   50|
|          9th|  514|
|    Bachelors| 5353|
|    Doctorate|  413|
|      HS-grad|10494|
|         11th| 1175|
| Some-college| 7282|
|      1st-4th|  166|
+-------------+-----+

+--------------------+-----+
|      marital_status|count|
+--------------------+-----+
|             Widowed|  993|
| Married-spouse-a...|  418|
|   Married-AF-spouse|   23|
|  Married-civ-spouse|14970|
|            Divo

The workclass and occupation columns have a significant number of missing values masked as "?", 1,843 in occupation and 1836 in the workclass.

Since "?" is essentially its own category where data wasn't captured, the safest and most transparent move is to explicitly replace it with the string "Unknown"

In [95]:

target_cols = ["workclass", "occupation"]

df_cleaned = df_clean
for col_name in target_cols:
    df_cleaned = df_cleaned.withColumn(
        col_name, 
        F.when(F.trim(F.col(col_name)) == "?", "Unknown").otherwise(F.col(col_name))
    )
df_cleaned.show(5)

+---+---------+--------+-------------+---------------+--------------------+------------------+---------------+------+-------+------------+------------+--------------+--------------+------------+
|age|workclass|  weight|    education|education_years|      marital_status|        occupation|   relationship|  race|    sex|capital_gain|capital_loss|hours_per_week|   citizenship|income_class|
+---+---------+--------+-------------+---------------+--------------------+------------------+---------------+------+-------+------------+------------+--------------+--------------+------------+
| 40|  Private|286370.0|      7th-8th|            4.0|  Married-civ-spouse| Machine-op-inspct|        Husband| White|   Male|         0.0|         0.0|          40.0|        Mexico|        >50K|
| 19|  Private|232392.0|      HS-grad|            9.0|       Never-married|     Other-service| Other-relative| White| Female|         0.0|         0.0|          40.0| United-States|       <=50K|
| 74|  Private| 99183.0| 

## Data Preprocessing

### Encoding categorical values

In [96]:
categorical_cols = ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'citizenship', 'income_class']
indexed_cols = ["workclass_indexed", "education_indexed", "marital_status_indexed", "occupation_indexed", "relationship_indexed", "race_indexed", "sex_indexed", "citizenship_indexed", "income_class_indexed"]


indexer = StringIndexer(inputCols=categorical_cols, outputCols=indexed_cols)
indexed_df = indexer.fit(df_cleaned).transform(df_cleaned)
indexed_df.show(5)

+---+---------+--------+-------------+---------------+--------------------+------------------+---------------+------+-------+------------+------------+--------------+--------------+------------+-----------------+-----------------+----------------------+------------------+--------------------+------------+-----------+-------------------+--------------------+
|age|workclass|  weight|    education|education_years|      marital_status|        occupation|   relationship|  race|    sex|capital_gain|capital_loss|hours_per_week|   citizenship|income_class|workclass_indexed|education_indexed|marital_status_indexed|occupation_indexed|relationship_indexed|race_indexed|sex_indexed|citizenship_indexed|income_class_indexed|
+---+---------+--------+-------------+---------------+--------------------+------------------+---------------+------+-------+------------+------------+--------------+--------------+------------+-----------------+-----------------+----------------------+------------------+--------

### Feature Assembling

In [97]:
feature_cols = ["age", "weight", "education_years", "capital_gain", "capital_loss", "hours_per_week", "workclass_indexed", "education_indexed", "marital_status_indexed", "occupation_indexed", "relationship_indexed", "race_indexed", "sex_indexed", "citizenship_indexed"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features") 
assembled_df = assembler.transform(indexed_df)
assembled_df.show(5)

+---+---------+--------+-------------+---------------+--------------------+------------------+---------------+------+-------+------------+------------+--------------+--------------+------------+-----------------+-----------------+----------------------+------------------+--------------------+------------+-----------+-------------------+--------------------+--------------------+
|age|workclass|  weight|    education|education_years|      marital_status|        occupation|   relationship|  race|    sex|capital_gain|capital_loss|hours_per_week|   citizenship|income_class|workclass_indexed|education_indexed|marital_status_indexed|occupation_indexed|relationship_indexed|race_indexed|sex_indexed|citizenship_indexed|income_class_indexed|            features|
+---+---------+--------+-------------+---------------+--------------------+------------------+---------------+------+-------+------------+------------+--------------+--------------+------------+-----------------+-----------------+--------

In [98]:
#Finalize the output dataframe with only "features" and "income_class_indexed" columns
final_output = assembled_df.select("features", "income_class_indexed")
final_output.show(5, truncate=False)

+-----------------------------------------------------------------+--------------------+
|features                                                         |income_class_indexed|
+-----------------------------------------------------------------+--------------------+
|(14,[0,1,2,5,7,9,13],[40.0,286370.0,4.0,40.0,8.0,6.0,1.0])       |1.0                 |
|[19.0,232392.0,9.0,0.0,0.0,40.0,0.0,0.0,1.0,5.0,5.0,0.0,1.0,0.0] |0.0                 |
|[74.0,99183.0,10.0,0.0,0.0,9.0,0.0,1.0,2.0,3.0,1.0,0.0,1.0,0.0]  |0.0                 |
|[50.0,23780.0,14.0,0.0,0.0,40.0,3.0,3.0,5.0,7.0,5.0,0.0,0.0,0.0] |0.0                 |
|[22.0,137510.0,10.0,0.0,0.0,40.0,0.0,1.0,1.0,3.0,2.0,0.0,0.0,0.0]|0.0                 |
+-----------------------------------------------------------------+--------------------+
only showing top 5 rows


## Modeling

### Data Splitting

In [99]:
#split the data into train and test sets
train_data, test_data = final_output.randomSplit([0.8, 0.2], seed=42)
print("Training Data Count: ", train_data.count())
print("Test Data Count: ", test_data.count())

Training Data Count:  26165
Test Data Count:  6372


### Decision Tree Classifier

In [ ]:
#descion tree model
dt = DecisionTreeClassifier(featuresCol="features", labelCol="income_class_indexed", seed=42, maxBins=50) #random_state is called seed in spark
dt_model = dt.fit(train_data)

#### Decision Tree Classifier Testing

In [101]:
#Test the model on the test data
dt_predictions = dt_model.transform(test_data)
dt_predictions.show(5, truncate=False)

+--------------------------------------------------------------+--------------------+-------------+------------------------------------------+----------+
|features                                                      |income_class_indexed|rawPrediction|probability                               |prediction|
+--------------------------------------------------------------+--------------------+-------------+------------------------------------------+----------+
|(14,[0,1,2,3,5],[41.0,163287.0,9.0,7688.0,43.0])              |1.0                 |[0.0,308.0]  |[0.0,1.0]                                 |1.0       |
|(14,[0,1,2,3,5,6],[36.0,61778.0,9.0,15024.0,40.0,2.0])        |1.0                 |[0.0,308.0]  |[0.0,1.0]                                 |1.0       |
|(14,[0,1,2,3,5,6,7],[34.0,34104.0,14.0,7688.0,38.0,4.0,3.0])  |1.0                 |[1.0,510.0]  |[0.0019569471624266144,0.9980430528375733]|1.0       |
|(14,[0,1,2,3,5,6,7],[40.0,153031.0,14.0,7298.0,35.0,2.0,3.0]) |1.0         

#### Decision Tree Classifier Evaluation 

In [102]:
#accuracy evaluation
evaluator = MulticlassClassificationEvaluator(labelCol="income_class_indexed", predictionCol="prediction", metricName="accuracy")
dt_accuracy = evaluator.evaluate(dt_predictions)
print("Decision Tree Test Accuracy = ", dt_accuracy)

Decision Tree Test Accuracy =  0.8473006905210295


### Random Forest Classifier

In [103]:
#random forest model
rf = RandomForestClassifier(featuresCol="features", labelCol="income_class_indexed",  maxBins=50, seed=42)
rf_model = rf.fit(train_data)

#### Random Forest Classifier Testing

In [104]:
#Test the model on the test data
rf_predictions = rf_model.transform(test_data)
rf_predictions.select("features", "income_class_indexed", "prediction").show(5, truncate=False)

+--------------------------------------------------------------+--------------------+----------+
|features                                                      |income_class_indexed|prediction|
+--------------------------------------------------------------+--------------------+----------+
|(14,[0,1,2,3,5],[41.0,163287.0,9.0,7688.0,43.0])              |1.0                 |1.0       |
|(14,[0,1,2,3,5,6],[36.0,61778.0,9.0,15024.0,40.0,2.0])        |1.0                 |1.0       |
|(14,[0,1,2,3,5,6,7],[34.0,34104.0,14.0,7688.0,38.0,4.0,3.0])  |1.0                 |1.0       |
|(14,[0,1,2,3,5,6,7],[40.0,153031.0,14.0,7298.0,35.0,2.0,3.0]) |1.0                 |1.0       |
|(14,[0,1,2,3,5,6,7],[45.0,181307.0,15.0,15024.0,43.0,5.0,9.0])|1.0                 |1.0       |
+--------------------------------------------------------------+--------------------+----------+
only showing top 5 rows


#### Random Forest Classifier Evaluation

In [105]:
#evaluate random forest accuracy
rf_accuracy = evaluator.evaluate(rf_predictions)
print("Random Forest Test Accuracy = ", rf_accuracy)

Random Forest Test Accuracy =  0.8537350910232266


In [106]:
#terminate the spark session
spark.stop()